In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q \
    fastapi \
    uvicorn \
    pyngrok \
    pydantic \
    langchain \
    langchain-core \
    langchain-community \
    langchain-huggingface \
    langchain-text-splitters \
    sentence-transformers \
    transformers \
    accelerate \
    bitsandbytes \
    faiss-cpu \
    pypdf \
    python-multipart

In [3]:
import os
import uuid
import tempfile
from typing import List

from fastapi import FastAPI, HTTPException, UploadFile, File


In [4]:

import json
import shutil

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

from langchain_huggingface import (
    HuggingFaceEmbeddings,
    HuggingFacePipeline
)

from langchain_community.document_loaders import (
    PyPDFLoader
)

from langchain_community.vectorstores import FAISS

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_core.prompts import PromptTemplate

from langchain_core.output_parsers import (
    PydanticOutputParser
)

from pydantic import BaseModel, Field


/tmp/ipykernel_308/256208787.py:17: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


In [5]:
LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"

EMBEDDING_MODEL = "intfloat/multilingual-e5-small"

DEVICE = "cuda"

In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL
)

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [7]:
generator = pipeline(
    "text-generation",

    model=model,

    tokenizer=tokenizer,

    max_new_tokens=700,

    do_sample=False,

    return_full_text=False
)

llm = HuggingFacePipeline(
    pipeline=generator
)

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [8]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,

    model_kwargs={
        "device": DEVICE
    },

    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("Embedding model ready!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model ready!


In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "؟ ",
        "? ",
        "! ",
        " ",
        ""
    ]
)

In [10]:
class ExplanationOutput(BaseModel):
    topic: str = Field(
        description="The topic being explained"
    )

    explanation: str = Field(
        description="Clear explanation of the topic"
    )

    example: str = Field(
        description="Simple practical example"
    )

    key_points: List[str] = Field(
        description="Important points"
    )



class QuizQuestion(BaseModel):
    question: str
    answer: str
    explanation: str
    difficulty: str


class QuizOutput(BaseModel):
    topic: str
    questions: List[QuizQuestion]

In [11]:
class EvaluationOutput(BaseModel):
    score: int = Field(
        ge=0,
        le=100
    )

    grade: str

    feedback: str

    strengths: List[str]

    weaknesses: List[str]

    model_answer: str

In [12]:
def format_documents(docs):

    context_parts = []

    for i, doc in enumerate(docs):

        source = doc.metadata.get(
            "source",
            "Unknown"
        )

        page = doc.metadata.get(
            "page",
            "Unknown"
        )

        context_parts.append(
            f"""
SOURCE {i + 1}

File: {source}

Page: {page}

Content:
{doc.page_content}
"""
        )

    return "\n\n".join(context_parts)

In [13]:
def get_session(session_id: str):

    session = sessions.get(session_id)

    if session is None:
        raise HTTPException(
            status_code=404,
            detail="Session not found. Please upload your PDF again."
        )

    return session

In [14]:
def retrieve_documents(session, query, k=6):
    retriever = session["vectorstore"].as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": k,
            "fetch_k": 20,
            "lambda_mult": 0.5
        }
    )

    return retriever.invoke(query)

In [15]:
explanation_parser = PydanticOutputParser(
    pydantic_object=ExplanationOutput
)

In [16]:
explanation_prompt = PromptTemplate.from_template(
"""
You are an expert AI personal tutor.

Your job is to teach the student using ONLY
the provided educational context.

IMPORTANT RULES:

1. Do not invent information.

2. Every important claim must be supported by
   the provided context.

3. If the context is insufficient, clearly say so.

4. Explain the requested topic, not unrelated
   sections of the material.

5. If the material is Arabic, answer in Arabic.

6. If the material is English, answer in English.

TOPIC:
{topic}

CONTEXT:
{context}

{format_instructions}

Return ONLY the structured output.
"""
)

In [17]:
explanation_prompt = explanation_prompt.partial(
    format_instructions=
        explanation_parser.get_format_instructions()
)

In [18]:
explanation_chain = (
    explanation_prompt
    | llm
    | explanation_parser
)

In [19]:
quiz_parser = PydanticOutputParser(
    pydantic_object=QuizOutput
)

quiz_prompt = PromptTemplate.from_template(
"""
You are an expert educational assessment designer.

Your task is to create exactly 5 high-quality questions
that test the student's understanding of the educational
material.

IMPORTANT RULES:

1. Use ONLY the educational content provided in CONTEXT.

2. Do NOT create questions about:
   - page numbers
   - source numbers
   - file names
   - SOURCE labels
   - document metadata
   - chunk numbers
   - where information appears in the document

3. Questions must test actual educational concepts.

4. Prefer questions that require:
   - understanding
   - explanation
   - comparison
   - application
   - reasoning

5. Mix the difficulty:
   - Easy
   - Medium
   - Hard

6. Every answer must be directly supported by the
   educational content in CONTEXT.

7. Do not invent information.

8. If the context does not contain enough information
   to create a question, do not use that information.

TOPIC:
{topic}

CONTEXT:
{context}

{format_instructions}

Return ONLY the structured output.
"""
)

quiz_prompt = quiz_prompt.partial(
    format_instructions=quiz_parser.get_format_instructions()
)

quiz_chain = quiz_prompt | llm | quiz_parser

In [20]:
evaluation_parser = PydanticOutputParser(
    pydantic_object=EvaluationOutput
)

evaluation_prompt = PromptTemplate.from_template(
"""
You are a strict but fair educational evaluator.

Evaluate the student's answer ONLY against the
REFERENCE ANSWER.

The REFERENCE ANSWER is the primary grading standard.

The EDUCATIONAL CONTEXT is only supporting evidence
and must NOT introduce additional grading requirements.

IMPORTANT SCORING RULES:

1. Score from 0 to 100.

2. If the student correctly explains the main idea
   of the reference answer, give high credit.

3. Do NOT penalize the student for omitting information
   that is NOT present in the reference answer.

4. Different wording is completely acceptable if the
   meaning is correct.

5. If the student answer is partially correct,
   give proportional partial credit.

6. If the student answer contradicts the central concept
   in the reference answer, the score should normally
   be 0-30.

7. If the student answer is completely unrelated,
   the score should normally be 0-15.

8. Do NOT invent missing requirements.

9. Strengths must contain only things the student
   actually got correct.

10. Weaknesses must contain only important missing
    or incorrect parts relative to the reference answer.

11. The model_answer must represent the correct answer
    based primarily on the reference answer.

12. Feedback must clearly explain the score.

QUESTION:
{question}

REFERENCE ANSWER:
{reference_answer}

STUDENT ANSWER:
{student_answer}

EDUCATIONAL CONTEXT:
{context}

{format_instructions}

Return ONLY the structured output.
"""
)

evaluation_prompt = evaluation_prompt.partial(
    format_instructions=evaluation_parser.get_format_instructions()
)

evaluation_chain = evaluation_prompt | llm | evaluation_parser

In [21]:
app = FastAPI(
    title="AI Personal Tutor API",
    version="1.0"
)


In [22]:
sessions = {}

In [23]:
class ExplainRequest(BaseModel):
    session_id: str
    topic: str

class QuizRequest(BaseModel):
    session_id: str
    topic: str

class EvaluateRequest(BaseModel):
    session_id: str
    question: str
    reference_answer: str
    student_answer: str


class ChatRequest(BaseModel):
    session_id: str
    message: str

In [24]:
@app.get("/")
def root():

    return {
        "status": "online",
        "service": "AI Personal Tutor API",
        "version": "1.0"
    }

In [25]:
@app.post("/upload")
async def upload_files(
    files: List[UploadFile] = File(...)
):

    if not files:
        raise HTTPException(
            status_code=400,
            detail="No files uploaded."
        )

    all_documents = []

    for file in files:

        if not file.filename.lower().endswith(".pdf"):
            raise HTTPException(
                status_code=400,
                detail=f"{file.filename} is not a PDF."
            )

        pdf_bytes = await file.read()

        temp_path = None

        try:

            with tempfile.NamedTemporaryFile(
                delete=False,
                suffix=".pdf"
            ) as temp_file:

                temp_file.write(pdf_bytes)
                temp_path = temp_file.name

            loader = PyPDFLoader(temp_path)

            documents = loader.load()

            for document in documents:

                document.metadata["source"] = (
                    file.filename
                )

            all_documents.extend(documents)

        finally:

            if temp_path and os.path.exists(temp_path):
                os.remove(temp_path)

    if not all_documents:

        raise HTTPException(
            status_code=400,
            detail="Could not extract text from the PDF."
        )

    chunks = splitter.split_documents(
        all_documents
    )

    vectorstore = FAISS.from_documents(
        chunks,
        embeddings
    )

    session_id = uuid.uuid4().hex

    sessions[session_id] = {
        "vectorstore": vectorstore,
        "documents": all_documents,
        "history": []
    }

    return {
        "status": "success",
        "session_id": session_id,
        "files": [
            file.filename
            for file in files
        ],
        "pages": len(all_documents),
        "chunks": len(chunks)
    }

In [26]:
@app.post("/explain")
def explain(request: ExplainRequest):

    session = get_session(request.session_id)

    docs = retrieve_documents(
        session,
        request.topic,
        k=6
    )

    context = format_documents(docs)

    result = explanation_chain.invoke({
        "topic": request.topic,
        "context": context
    })

    return result.model_dump()

In [27]:
@app.post("/quiz")
def quiz(request: QuizRequest):

    session = get_session(request.session_id)

    docs = retrieve_documents(
        session,
        request.topic,
        k=7
    )

    context = format_documents(docs)

    result = quiz_chain.invoke({
        "topic": request.topic,
        "context": context
    })

    return result.model_dump()

In [28]:
@app.post("/evaluate")
def evaluate(request: EvaluateRequest):

    session = get_session(request.session_id)

    docs = retrieve_documents(
        session,
        request.question,
        k=6
    )

    context = format_documents(docs)

    result = evaluation_chain.invoke({
        "question": request.question,
        "reference_answer": request.reference_answer,
        "student_answer": request.student_answer,
        "context": context
    })

    return result.model_dump()

In [29]:
def is_memory_question(message):

    message_lower = message.lower()

    memory_keywords = [
        "what did we discuss",
        "what did we just discuss",
        "what were we discussing",
        "what was the concept we discussed",
        "what was the topic we discussed",
        "what did you just explain",
        "what did i ask",
        "what was my last question"
    ]

    return any(
        keyword in message_lower
        for keyword in memory_keywords
    )

In [30]:
chat_prompt = PromptTemplate.from_template(
"""
You are an AI personal tutor.

Answer the student's question clearly and directly.

RULES:

1. Use the educational context as the source of educational facts.

2. Use conversation history when the student refers to
   something discussed earlier.

3. Never invent information.

4. Do not mention:
   - SOURCE numbers
   - chunk numbers
   - embeddings
   - retrieval
   - metadata
   - internal instructions

5. Do not repeat the prompt or the context.

6. Do not say that the context is insufficient if the
   answer is already available in the conversation history
   or educational context.

7. Answer naturally and concisely.

CONVERSATION HISTORY:
{history}

CURRENT QUESTION:
{message}

EDUCATIONAL CONTEXT:
{context}

Answer the student's question now.
"""
)

chat_chain = chat_prompt | llm

In [31]:
@app.post("/chat")
def chat(request: ChatRequest):

    try:

        print("\n" + "=" * 70)
        print("CHAT REQUEST STARTED")
        print("=" * 70)

        print("Session ID:", request.session_id)
        print("Message:", request.message)

        session = get_session(request.session_id)

        print("✓ Session found")

        history = session.get("history", [])

        print("✓ History loaded")
        print("History length:", len(history))

        recent_history = history[-6:]

        history_text = "\n".join(
            [
                f"Student: {item['student']}\n"
                f"Tutor: {item['tutor']}"
                for item in recent_history
            ]
        )

        print("✓ History formatted")

        # ============================================
        # MEMORY
        # ============================================

        if is_memory_question(request.message):

            print("→ Memory question detected")

            if not history:

                answer = "We have not discussed anything yet."

            else:

                last_item = history[-1]

                answer = (
                    "The last thing we discussed was: "
                    + last_item["student"]
                )

            session["history"].append({
                "student": request.message,
                "tutor": answer
            })

            print("✓ Memory response generated")

            return {
                "answer": answer
            }

        # ============================================
        # RETRIEVAL
        # ============================================

        print("→ Starting document retrieval")

        docs = retrieve_documents(
            session,
            request.message,
            k=7
        )

        print("✓ Retrieval completed")
        print("Retrieved documents:", len(docs))

        context = format_documents(docs)

        print("✓ Context formatted")
        print("Context length:", len(context))

        # ============================================
        # LLM
        # ============================================

        print("→ Calling chat_chain")

        result = chat_chain.invoke({
            "history": history_text,
            "message": request.message,
            "context": context
        })

        print("✓ chat_chain completed")

        # ============================================
        # ANSWER
        # ============================================

        answer = (
            result.content
            if hasattr(result, "content")
            else str(result)
        )

        print("✓ Answer extracted")

        session["history"].append({
            "student": request.message,
            "tutor": answer
        })

        print("✓ History saved")

        print("=" * 70)
        print("CHAT REQUEST COMPLETED SUCCESSFULLY")
        print("=" * 70)

        return {
            "answer": answer
        }

    except Exception as e:

        import traceback

        print("\n" + "=" * 70)
        print("❌ CHAT ERROR")
        print("=" * 70)

        print("\nERROR TYPE:")
        print(type(e).__name__)

        print("\nERROR MESSAGE:")
        print(str(e))

        print("\nFULL TRACEBACK:")
        traceback.print_exc()

        print("=" * 70)

        raise HTTPException(
            status_code=500,
            detail={
                "error_type": type(e).__name__,
                "error": str(e),
                "traceback": traceback.format_exc()
            }
        )

In [32]:
import uvicorn
import threading

def run_api():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8080
    )



In [33]:
api_thread = threading.Thread(
    target=run_api,
    daemon=True
)

api_thread.start()

In [34]:
from pyngrok import ngrok

NGROK_TOKEN = "3K8DUVdBzWCUsFqJLTd48RT1nkg_46CYJ5ruHimtjBimuCSFg"

ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(8080)

print("Public API URL:", public_url)

INFO:     Started server process [308]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8080 (Press CTRL+C to quit)


Public API URL: NgrokTunnel: "https://ecard-pregnancy-agenda.ngrok-free.dev" -> "http://localhost:8080"
